In [2]:
import arcpy

In [3]:
import math
import arcpy

project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap

if active_map is None:
    raise RuntimeError("Activate the Missouri River Flood 2019 map and rerun the cell.")

october_layer = active_map.listLayers("gtc_20181006.crf")[0]
april_layer = active_map.listLayers("gtc_aligned_20190404.crf")[0]

october = arcpy.Raster(october_layer.dataSource)
april = arcpy.Raster(april_layer.dataSource)

for label, raster in [("October", october), ("Aligned April", april)]:
    print(f"{label} | EPSG:{raster.spatialReference.factoryCode}")
    print(f"  Columns × rows: {raster.width:,} × {raster.height:,}")
    print(f"  Cell size X/Y:  {raster.meanCellWidth:.15g}, {raster.meanCellHeight:.15g}")
    print(f"  Left / Bottom:  {raster.extent.XMin:.12f}, {raster.extent.YMin:.12f}")

x_offset = (april.extent.XMin - october.extent.XMin) / october.meanCellWidth
y_offset = (april.extent.YMin - october.extent.YMin) / october.meanCellHeight

same_crs = (
    october.spatialReference.factoryCode
    == april.spatialReference.factoryCode
    == 4326
)
same_cell_size = (
    math.isclose(april.meanCellWidth, october.meanCellWidth, rel_tol=1e-9)
    and math.isclose(april.meanCellHeight, october.meanCellHeight, rel_tol=1e-9)
)
whole_cell_offsets = all(
    math.isclose(offset, round(offset), rel_tol=0, abs_tol=1e-6)
    for offset in (x_offset, y_offset)
)

print(f"\nApril offset in pixels (X, Y): ({x_offset:.9f}, {y_offset:.9f})")
print(f"Both EPSG:4326: {same_crs}")
print(f"Matching cell sizes: {same_cell_size}")
print(f"Whole-cell offsets: {whole_cell_offsets}")

aligned = same_crs and same_cell_size and whole_cell_offsets
print("PASS — grids align" if aligned else "CHECK — grids differ")

October | EPSG:4326
  Columns × rows: 28,709 × 16,757
  Cell size X/Y:  0.000120476645244977, 0.000120476645244977
  Left / Bottom:  -98.049111522110, 39.580791687244
Aligned April | EPSG:4326
  Columns × rows: 28,705 × 16,758
  Cell size X/Y:  0.000120476645244977, 0.000120476645244977
  Left / Bottom:  -98.049111522110, 39.580791687244

April offset in pixels (X, Y): (0.000000000, 0.000000000)
Both EPSG:4326: True
Matching cell sizes: True
Whole-cell offsets: True
PASS — grids align
